[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tunnel-ai/way/blob/main/notebooks/06_01_main_classical.ipynb)

# Module 6, Vision: When Humans Design the Features

**Notebook:** `06_01_main_classical`

## Where this fits

In the previous notebook, we saw that a visual pattern can be extracted by applying a filter to local regions of an image.

Now we ask:

> **What if humans decide which visual properties matter, measure those properties, and then use an ordinary machine-learning model?**

That is the basic classical computer-vision recipe:

\[
\text{image}
\rightarrow
\text{human-designed features}
\rightarrow
\text{tabular model}
\rightarrow
\text{prediction}
\]

We will use **EuroSAT**, a 10-class satellite-imagery dataset.

Our feature set is deliberately simple:

- color statistics;
- color histograms;
- Sobel edge summaries.

This is **not** meant to reproduce the strongest pre-deep-learning vision systems. Classical computer vision also used much richer descriptors such as HOG, SIFT, SURF, and texture features.

The point here is more fundamental:

> **Every feature in this notebook is something a human chose and can name.**

That sets up the contrast with the next notebook, where a CNN learns its own visual features from data.

## 0) Setup

The notebook loads EuroSAT from a hosted Hugging Face dataset and runs independently in a fresh Colab session.

We use:

- NumPy and PIL for image data;
- SciPy for Sobel filtering;
- scikit-learn for the classifiers and evaluation.

The first run downloads the hosted EuroSAT image data (roughly 90 MB).

In [ ]:
import sys
import subprocess
import importlib.util

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image
from scipy.ndimage import convolve

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
)

if importlib.util.find_spec("datasets") is None:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", "datasets"]
    )

from datasets import load_dataset, concatenate_datasets

SEED = 1955
N_SAMPLES = 5000

print("Setup complete.")


## 1) Load EuroSAT

EuroSAT contains 27,000 RGB satellite tiles across 10 land-cover classes.

To keep the walkthrough fast, we use a deterministic sample of 5,000 images.

### Why deterministic?

All of the vision notebooks will use the **same sampling and split logic**:

1. sort all image paths;
2. draw the same seeded sample;
3. create the same 70/10/20 train/validation/test split.

That means later comparisons among:

- hand-engineered features;
- a CNN trained from scratch;
- transfer learning;

are based on the same underlying observations.

Each notebook still runs independently.

In [ ]:
HF_DATASET = "giswqs/EuroSAT_RGB"

print("Loading EuroSAT from Hugging Face...")
dataset_dict = load_dataset(HF_DATASET)

# Combine the hosted splits because we create our own deterministic course split.
full_dataset = concatenate_datasets(
    [
        dataset_dict["train"],
        dataset_dict["validation"],
        dataset_dict["test"],
    ]
).sort("filename")

label_names = list(
    full_dataset.features["label"].names
)
num_classes = len(label_names)

print(f"Hosted images: {len(full_dataset):,}")
print("Classes:", label_names)

# Deterministic sample from filenames sorted above.
rng = np.random.default_rng(SEED)
sample_idx = rng.choice(
    len(full_dataset),
    size=N_SAMPLES,
    replace=False,
)

sampled = full_dataset.select(
    sample_idx.tolist()
)

y = np.asarray(
    sampled["label"],
    dtype=np.int64,
)

idx_all = np.arange(N_SAMPLES)

idx_dev, idx_test = train_test_split(
    idx_all,
    test_size=0.20,
    stratify=y,
    random_state=SEED,
)

idx_train, idx_val = train_test_split(
    idx_dev,
    test_size=0.125,
    stratify=y[idx_dev],
    random_state=SEED,
)

X_img = np.empty(
    (N_SAMPLES, 64, 64, 3),
    dtype=np.uint8,
)

for i, row in enumerate(sampled):
    X_img[i] = np.asarray(
        row["image"].convert("RGB"),
        dtype=np.uint8,
    )

print(
    f"train={len(idx_train):,}  "
    f"validation={len(idx_val):,}  "
    f"test={len(idx_test):,}"
)
print("Image array:", X_img.shape, X_img.dtype)

counts = pd.Series(
    [label_names[i] for i in y]
).value_counts().sort_index()

print("\nSampled class counts:")
print(counts)


### Build one deterministic 5,000-image sample

We sort the filenames before sampling. That small detail matters: a random seed cannot guarantee reproducibility if the input ordering itself changes.

Then we create:

- 70% training;
- 10% validation;
- 20% test.

The classical models will not need the validation set much, but keeping it here makes the data structure consistent with the CNN notebooks.

### Look at the task

To a human, several classes have obvious visual cues:

- `Forest` tends to be green and textured;
- `SeaLake` often has large blue regions;
- `Residential` contains repeated built structures;
- `Highway` often contains long linear structures.

But notice the challenge:

> Some distinctions depend on **what colors are present**, while others depend on **how visual elements are arranged in space**.

Our hand-engineered baseline will be much better at the first problem than the second.

In [ ]:
fig, axs = plt.subplots(2, 5, figsize=(13, 6))

for class_id, ax in zip(range(num_classes), axs.flat):
    i = np.where(y == class_id)[0][0]

    ax.imshow(X_img[i])
    ax.set_title(label_names[class_id], fontsize=10)
    ax.axis("off")

plt.tight_layout()
plt.show()

## 2) Engineer visual features

We will create **34 features per image** in three blocks.

### A. Color statistics — 6 features

For red, green, and blue:

- mean intensity;
- standard deviation.

These answer questions such as:

> How green is this image overall?  
> How variable are its colors?

### B. Color histograms — 24 features

For each color channel, count the fraction of pixels falling into eight intensity ranges.

This captures the **distribution** of color rather than only its average.

### C. Edge summaries — 4 features

Use the Sobel filters from the intuition notebook and summarize the average horizontal and vertical edge response in the top and bottom halves of the image.

These provide a crude measure of visual structure.

Every one of these 34 columns was chosen by us.

In [ ]:
def color_stats(img_batch):
    """Per-channel mean and standard deviation. Returns (N, 6)."""

    x = img_batch.astype(np.float32) / 255.0

    means = x.mean(axis=(1, 2))
    stds = x.std(axis=(1, 2))

    return np.concatenate(
        [means, stds],
        axis=1,
    )


feat_color = color_stats(X_img)
print("Color-statistics block:", feat_color.shape)

In [ ]:
def color_histograms(img_batch, bins=8):
    """Normalized per-channel histograms. Returns (N, 3*bins)."""

    n = img_batch.shape[0]
    output = np.zeros(
        (n, 3 * bins),
        dtype=np.float32,
    )

    bin_edges = np.linspace(
        0,
        256,
        bins + 1,
    )

    for i in range(n):
        for channel in range(3):
            hist, _ = np.histogram(
                img_batch[i, ..., channel],
                bins=bin_edges,
            )

            output[
                i,
                channel * bins : (channel + 1) * bins,
            ] = hist / hist.sum()

    return output


feat_hist = color_histograms(
    X_img,
    bins=8,
)

print("Color-histogram block:", feat_hist.shape)

In [ ]:
SOBEL_X = np.array([
    [-1, 0, 1],
    [-2, 0, 2],
    [-1, 0, 1],
], dtype=np.float32)

SOBEL_Y = SOBEL_X.T


def edge_summaries(img_batch):
    """Mean absolute Sobel response in top/bottom halves. Returns (N, 4)."""

    gray = (
        img_batch.astype(np.float32).mean(axis=3)
        / 255.0
    )

    output = np.zeros(
        (gray.shape[0], 4),
        dtype=np.float32,
    )

    half = gray.shape[1] // 2

    for i in range(gray.shape[0]):
        gx = np.abs(
            convolve(
                gray[i],
                SOBEL_X,
                mode="nearest",
            )
        )

        gy = np.abs(
            convolve(
                gray[i],
                SOBEL_Y,
                mode="nearest",
            )
        )

        output[i] = [
            gx[:half].mean(),
            gy[:half].mean(),
            gx[half:].mean(),
            gy[half:].mean(),
        ]

    return output


feat_edges = edge_summaries(X_img)
print("Edge-summary block:", feat_edges.shape)

### Turn the images into a table

Once the features are extracted, the images themselves disappear from the modeling step.

Each image becomes one row with 34 named columns.

At this point, image classification looks like any other supervised tabular problem.

In [ ]:
X = np.concatenate(
    [
        feat_color,
        feat_hist,
        feat_edges,
    ],
    axis=1,
)

feature_names = (
    [f"mean_{c}" for c in "rgb"]
    + [f"std_{c}" for c in "rgb"]
    + [
        f"hist_{c}_{bin_id}"
        for c in "rgb"
        for bin_id in range(8)
    ]
    + [
        "edge_top_x",
        "edge_top_y",
        "edge_bottom_x",
        "edge_bottom_y",
    ]
)

features_df = pd.DataFrame(
    X,
    columns=feature_names,
)

print("Feature matrix:", X.shape)
features_df.head().round(3)

## 3) Train two ordinary classifiers

We will compare:

### Logistic regression

A linear baseline.

Each class receives a coefficient for every feature, so the model is relatively easy to inspect.

### Random forest

A nonlinear tabular model.

It can discover thresholds and interactions such as:

> high green + low blue + strong texture

without us explicitly specifying the combination.

Random guessing across 10 balanced-ish classes would be around 10%.

### Teaching note about the test set

We inspect test performance throughout this module because the goal is to compare modeling approaches interactively.

In a formal model-development project, repeatedly checking test results and then changing the model would gradually turn the test set into part of the development process. A truly untouched final evaluation set would normally be retained.

In [ ]:
X_train = X[idx_train]
X_val = X[idx_val]
X_test = X[idx_test]

y_train = y[idx_train]
y_val = y[idx_val]
y_test = y[idx_test]

# Logistic regression benefits from standardized features.
scaler = StandardScaler().fit(X_train)

X_train_s = scaler.transform(X_train)
X_val_s = scaler.transform(X_val)
X_test_s = scaler.transform(X_test)

print(
    f"train={X_train.shape}  "
    f"validation={X_val.shape}  "
    f"test={X_test.shape}"
)

In [ ]:
lr = LogisticRegression(
    max_iter=3000,
    C=1.0,
)

lr.fit(
    X_train_s,
    y_train,
)

rf = RandomForestClassifier(
    n_estimators=300,
    random_state=SEED,
    n_jobs=-1,
)

# Trees do not require standardization.
rf.fit(
    X_train,
    y_train,
)

lr_val_acc = accuracy_score(
    y_val,
    lr.predict(X_val_s),
)

rf_val_acc = accuracy_score(
    y_val,
    rf.predict(X_val),
)

lr_test_acc = accuracy_score(
    y_test,
    lr.predict(X_test_s),
)

rf_test_acc = accuracy_score(
    y_test,
    rf.predict(X_test),
)

print(f"Random chance baseline         : {1 / num_classes:.3f}")
print()
print(f"Logistic regression validation: {lr_val_acc:.3f}")
print(f"Random forest       validation: {rf_val_acc:.3f}")
print()
print(f"Logistic regression test      : {lr_test_acc:.3f}")
print(f"Random forest       test      : {rf_test_acc:.3f}")

### What should we conclude?

If these models perform substantially above 10%, then the hand-designed features contain real information about land-cover class.

But the important question is not just:

> **How accurate is the model?**

It is:

> **Which distinctions can these features represent, and which distinctions are they incapable of representing well?**

## 4) Which features is the random forest using?

Because our features have names, we can inspect feature importance.

Treat random-forest importance as a rough diagnostic rather than a causal explanation, but it can still tell us whether the model is leaning heavily on color, histograms, or edges.

In [ ]:
importance = pd.Series(
    rf.feature_importances_,
    index=feature_names,
).sort_values(ascending=False)

print("Top 12 random-forest features:")
print(importance.head(12).round(4))

importance.head(12).sort_values().plot(
    kind="barh",
    figsize=(8, 5),
    title="Top random-forest feature importances",
)

plt.xlabel("Feature importance")
plt.tight_layout()
plt.show()

## 5) Look beyond aggregate accuracy

Overall accuracy can hide very different performance across classes.

A useful vision workflow asks:

- Which classes are easy?
- Which classes are confused?
- Do the mistakes make sense given the representation?

We will use the random forest for the diagnostic analysis below.

In [ ]:
y_pred = rf.predict(X_test)

print(
    classification_report(
        y_test,
        y_pred,
        target_names=label_names,
        digits=3,
    )
)

### Confusion matrix

Pay particular attention to off-diagonal cells.

If two classes are often confused, ask whether our 34 features actually contain the information needed to distinguish them.

For example:

- color can help distinguish water from forest;
- simple edge density may identify a visually busy scene;
- but our feature table largely throws away **where** structures appear and how they are arranged.

That is likely to matter for distinctions such as roads, residential patterns, and industrial layouts.

In [ ]:
cm = confusion_matrix(
    y_test,
    y_pred,
)

fig, ax = plt.subplots(
    figsize=(9, 8),
)

im = ax.imshow(
    cm,
    cmap="Blues",
)

ax.set_xticks(range(num_classes))
ax.set_xticklabels(
    label_names,
    rotation=90,
    fontsize=9,
)

ax.set_yticks(range(num_classes))
ax.set_yticklabels(
    label_names,
    fontsize=9,
)

ax.set_xlabel("Predicted class")
ax.set_ylabel("True class")
ax.set_title("Random forest confusion matrix")

for i in range(num_classes):
    for j in range(num_classes):
        ax.text(
            j,
            i,
            cm[i, j],
            ha="center",
            va="center",
            fontsize=8,
            color=(
                "white"
                if cm[i, j] > cm.max() / 2
                else "black"
            ),
        )

plt.colorbar(
    im,
    ax=ax,
    fraction=0.046,
)

plt.tight_layout()
plt.show()

## 6) Inspect confident mistakes

A wrong prediction is especially interesting when the model is confident.

Those cases can reveal a mismatch between:

- what the model can measure;
- and what a human uses to recognize the scene.

Remember: the random forest never sees the image.

It only sees 34 numbers describing color and coarse edge statistics.

In [ ]:
probabilities = rf.predict_proba(X_test)
predictions = probabilities.argmax(axis=1)
confidence = probabilities.max(axis=1)

wrong = np.where(
    predictions != y_test
)[0]

wrong = wrong[
    np.argsort(
        -confidence[wrong]
    )
]

n_show = min(
    8,
    len(wrong),
)

selected = wrong[:n_show]

fig, axs = plt.subplots(
    2,
    4,
    figsize=(12, 6),
)

for ax in axs.flat:
    ax.axis("off")

for test_pos, ax in zip(
    selected,
    axs.flat,
):
    original_idx = idx_test[test_pos]

    ax.imshow(
        X_img[original_idx]
    )

    ax.set_title(
        f"true: {label_names[y_test[test_pos]]}\n"
        f"pred: {label_names[predictions[test_pos]]} "
        f"({confidence[test_pos]:.2f})",
        fontsize=9,
    )

    ax.axis("off")

plt.tight_layout()
plt.show()

## 7) What this baseline does—and does not—show

We built a useful classifier from only 34 human-designed measurements.

That establishes something important:

> **Images can be converted into tabular features, and ordinary machine learning can classify them.**

But our feature set is intentionally simple.

It captures:

- overall color;
- color distributions;
- coarse edge strength.

It does **not** capture spatial structure especially well.

A stronger classical computer-vision pipeline might add:

- HOG for local edge orientation;
- SIFT-like local descriptors;
- texture measures such as GLCM;
- spatial pyramids;
- more sophisticated feature aggregation.

So this notebook should not be interpreted as:

> “This is the best classical computer vision could do.”

It should be interpreted as:

> **“This is what happens when humans choose the representation.”**

## Takeaways

1. **Feature engineering converts vision into tabular data.**  
   Once we summarize each image with named measurements, ordinary classifiers work.

2. **Interpretability is a strength.**  
   We know exactly what `mean_g`, `hist_b_6`, or `edge_top_x` mean.

3. **Representation limits the model.**  
   If we do not encode an important visual property, the classifier cannot use it.

4. **Spatial structure is expensive to hand-design.**  
   We can keep inventing better descriptors, but the engineering burden grows.

---

## Where the next notebook goes

[`06_02_main_cnn.ipynb`](06_02_main_cnn.ipynb) keeps:

- the same EuroSAT task;
- the same deterministic 5,000-image sample;
- the same 70/10/20 split.

But it changes the representation strategy.

Instead of asking:

> **Which features should humans measure?**

we ask:

> **Can the network learn useful visual features directly from pixels?**

That is the central transition from classical computer vision to convolutional neural networks.